# Entrenamiento de M2 (Vehículos) y M3 (Keypoints de Ruedas) — Sin Roboflow

Este notebook permite crear tu dataset propio en **Google Drive** (sin Roboflow), pre-etiquetarlo automáticamente con **YOLOE**, entrenarlo en GPU T4 de Google Colab usando un **`venv`** limpio y exportar los pesos finales:
- `m2_seg.pt`: Segmentación de instancias (`0: motocicleta`, `1: automovil`, `2: minibus`, `3: camion`, `4: bus`).
- `m3_pose.pt`: Detección de llantas con **5 keypoints** (`0: centro`, `1: contacto`, `2: superior`, `3: izquierda`, `4: derecha`).

### Estructura esperada en tu Google Drive
Sube tus fotos propias a `MyDrive/peaje_datasets/raw_vehiculos/` (para M2) y `MyDrive/peaje_datasets/raw_laterales/` (fotos laterales para M3).


## 1. Montar Google Drive y crear entorno `venv`


In [ ]:
!nvidia-smi -L
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
BASE_DRIVE = Path("/content/drive/MyDrive/peaje_datasets")
PESOS_DRIVE = Path("/content/drive/MyDrive/peaje_modelos")
for d in (BASE_DRIVE / "raw_vehiculos", BASE_DRIVE / "raw_laterales", PESOS_DRIVE):
    d.mkdir(parents=True, exist_ok=True)

VENV_DIR = Path("/content/venv_train")
if not (VENV_DIR / "bin" / "python").exists():
    import subprocess, sys
    py_ver = f"{sys.version_info.major}.{sys.version_info.minor}"
    subprocess.run(["bash", "-c", f"apt-get -qq update && apt-get -qq install -y python{py_ver}-venv python3-venv >/dev/null 2>&1"], check=False)
    if subprocess.run([sys.executable, "-m", "venv", str(VENV_DIR)]).returncode != 0:
        subprocess.run([sys.executable, "-m", "venv", "--without-pip", str(VENV_DIR)], check=True)
        subprocess.run(["bash", "-c", f"curl -sS https://bootstrap.pypa.io/get-pip.py | '{VENV_DIR}/bin/python'"], check=True)
    !{VENV_DIR}/bin/pip install -q --upgrade pip
    !{VENV_DIR}/bin/pip install -q "ultralytics>=8.3.0" "opencv-python-headless>=4.11.0" "PyYAML>=6.0.2"
    !{VENV_DIR}/bin/pip install -q "git+https://github.com/ultralytics/CLIP.git" || true

VENV_PY = VENV_DIR / "bin" / "python"
print("Listo. Sube tus fotos a:", BASE_DRIVE)


## 2. Pre-etiquetado automático (Auto-labeling) de tus fotos para M2 y M3


In [ ]:
script_auto = Path("/content/auto_etiquetar.py")
script_auto.write_text('''
import random, shutil
from pathlib import Path
import cv2
import numpy as np
import yaml
from ultralytics import YOLOE

BASE = Path("/content/drive/MyDrive/peaje_datasets")
CLASES_M2 = ["motocicleta", "automovil", "minibus", "camion", "bus"]
PROMPTS_M2 = ["motorcycle", "car", "minibus", "truck", "bus"]
MAP_M2 = {"motorcycle": 0, "car": 1, "minibus": 2, "truck": 3, "bus": 4}

def preparar_dirs(raiz: Path):
    for sp in ("train", "val"):
        (raiz / "images" / sp).mkdir(parents=True, exist_ok=True)
        (raiz / "labels" / sp).mkdir(parents=True, exist_ok=True)

def auto_m2():
    imgs = sorted([p for p in (BASE / "raw_vehiculos").glob("*") if p.suffix.lower() in (".jpg", ".jpeg", ".png", ".webp")])
    if not imgs:
        print("M2: No hay imagenes en raw_vehiculos/.")
        return
    ds = BASE / "dataset_m2_seg"
    preparar_dirs(ds)
    m = YOLOE("yoloe-11s-seg.pt")
    m.set_classes(PROMPTS_M2, m.get_text_pe(PROMPTS_M2))
    random.seed(42); random.shuffle(imgs)
    corte = max(1, int(len(imgs) * 0.8))
    for idx, ruta in enumerate(imgs):
        sp = "train" if idx < corte else "val"
        im = cv2.imread(str(ruta))
        if im is None: continue
        h, w = im.shape[:2]
        r = m.predict(im, conf=0.30, verbose=False)[0]
        lineas = []
        if r.boxes is not None and r.masks is not None:
            for i, b in enumerate(r.boxes):
                cls_name = str(r.names[int(b.cls)]).lower()
                if cls_name not in MAP_M2 or i >= len(r.masks.xyn) or len(r.masks.xyn[i]) < 3:
                    continue
                coords = " ".join(f"{x:.6f} {y:.6f}" for x, y in r.masks.xyn[i])
                lineas.append(f"{MAP_M2[cls_name]} {coords}")
        shutil.copy2(ruta, ds / "images" / sp / ruta.name)
        (ds / "labels" / sp / f"{ruta.stem}.txt").write_text("\n".join(lineas), encoding="utf-8")
    yaml.safe_dump({"path": str(ds), "train": "images/train", "val": "images/val",
                    "names": {i: c for i, c in enumerate(CLASES_M2)}}, open(ds / "data.yaml", "w"))
    print(f"M2 pre-etiquetado listo en {ds} ({len(imgs)} imagenes). Revisa/corrige en CVAT o Label Studio antes de entrenar.")

def auto_m3():
    imgs = sorted([p for p in (BASE / "raw_laterales").glob("*") if p.suffix.lower() in (".jpg", ".jpeg", ".png", ".webp")])
    if not imgs:
        print("M3: No hay imagenes en raw_laterales/.")
        return
    ds = BASE / "dataset_m3_pose"
    preparar_dirs(ds)
    m = YOLOE("yoloe-11s-seg.pt")
    m.set_classes(["wheel", "tire"], m.get_text_pe(["wheel", "tire"]))
    random.seed(42); random.shuffle(imgs)
    corte = max(1, int(len(imgs) * 0.8))
    for idx, ruta in enumerate(imgs):
        sp = "train" if idx < corte else "val"
        im = cv2.imread(str(ruta))
        if im is None: continue
        h, w = im.shape[:2]
        r = m.predict(im, conf=0.15, imgsz=1024, verbose=False)[0]
        lineas = []
        if r.boxes is not None and r.masks is not None:
            for i, b in enumerate(r.boxes):
                if i >= len(r.masks.xy) or len(r.masks.xy[i]) < 3:
                    continue
                x1, y1, x2, y2 = b.xyxy[0].tolist()
                xc, yc, bw, bh = ((x1+x2)/2)/w, ((y1+y2)/2)/h, (x2-x1)/w, (y2-y1)/h
                pts = np.asarray(r.masks.xy[i], dtype=np.float32)
                kps = [
                    (pts[:,0].mean()/w, pts[:,1].mean()/h, 2),
                    (pts[np.argmax(pts[:,1]),0]/w, pts[np.argmax(pts[:,1]),1]/h, 2),
                    (pts[np.argmin(pts[:,1]),0]/w, pts[np.argmin(pts[:,1]),1]/h, 2),
                    (pts[np.argmin(pts[:,0]),0]/w, pts[np.argmin(pts[:,0]),1]/h, 2),
                    (pts[np.argmax(pts[:,0]),0]/w, pts[np.argmax(pts[:,0]),1]/h, 2),
                ]
                kp_str = " ".join(f"{kx:.6f} {ky:.6f} {v}" for kx, ky, v in kps)
                lineas.append(f"0 {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f} {kp_str}")
        shutil.copy2(ruta, ds / "images" / sp / ruta.name)
        (ds / "labels" / sp / f"{ruta.stem}.txt").write_text("\n".join(lineas), encoding="utf-8")
    yaml.safe_dump({"path": str(ds), "train": "images/train", "val": "images/val",
                    "kpt_shape": [5, 3], "flip_idx": [0, 1, 2, 4, 3], "names": {0: "llanta"}},
                   open(ds / "data.yaml", "w"))
    print(f"M3 pre-etiquetado listo en {ds} ({len(imgs)} imagenes).")

if __name__ == "__main__":
    auto_m2()
    auto_m3()
''', encoding="utf-8")

!{VENV_PY} "{script_auto}"


## 3. Entrenamiento de M2 (`yolo11s-seg.pt` → `m2_seg.pt`) y M3 (`yolo11s-pose.pt` → `m3_pose.pt`)


In [ ]:
import shutil
from pathlib import Path

DS_M2 = BASE_DRIVE / "dataset_m2_seg" / "data.yaml"
DS_M3 = BASE_DRIVE / "dataset_m3_pose" / "data.yaml"

if DS_M2.is_file():
    !{VENV_PY} -m ultralytics segment train model=yolo11s-seg.pt data="{DS_M2}" epochs=50 imgsz=640 batch=16 project=/content/runs name=m2_vehiculos
    best_m2 = Path("/content/runs/m2_vehiculos/weights/best.pt")
    if best_m2.is_file():
        shutil.copy2(best_m2, PESOS_DRIVE / "m2_seg.pt")
        print("Peso M2 guardado en Drive:", PESOS_DRIVE / "m2_seg.pt")

if DS_M3.is_file():
    !{VENV_PY} -m ultralytics pose train model=yolo11s-pose.pt data="{DS_M3}" epochs=60 imgsz=640 batch=16 project=/content/runs name=m3_ruedas
    best_m3 = Path("/content/runs/m3_ruedas/weights/best.pt")
    if best_m3.is_file():
        shutil.copy2(best_m3, PESOS_DRIVE / "m3_pose.pt")
        print("Peso M3 guardado en Drive:", PESOS_DRIVE / "m3_pose.pt")
